In [2]:
#importing libraries
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

INPUT_FILE = "coverage_results_rich.json"

# -------- LOAD RESULTS --------
with open(INPUT_FILE, "r") as f:
    payload = json.load(f)

if isinstance(payload, list):
    data = payload
else:
    data = payload["dataset_results"]

df = pd.DataFrame(data)

print("\n✅ Loaded coverage results")
print(df.head())


# MAIN COLUMNS (t = 0.7)

title_col = "title_coverage_0.7"
description_col = "description_coverage_0.7"

# SUMMARY STATISTICS

print("\n📊 SUMMARY STATISTICS (t=0.7)\n")

summary = {
    "Metric": ["Mean", "Median", "Std", "Min", "Max"],

    "Title Coverage (0.7)": [
        df[title_col].mean(),
        df[title_col].median(),
        df[title_col].std(),
        df[title_col].min(),
        df[title_col].max(),
    ],

    "Description Coverage (0.7)": [
        df[description_col].mean(),
        df[description_col].median(),
        df[description_col].std(),
        df[description_col].min(),
        df[description_col].max(),
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df)

# HISTOGRAMS

# -------- Title vs Description --------
plt.figure()

plt.hist(
    df[title_col],
    bins=50,
    alpha=0.5,
    label="Title"
)

plt.hist(
    df[description_col],
    bins=50,
    alpha=0.5,
    label="Description"
)

plt.xlabel("Coverage")
plt.ylabel("Number of Datasets")

plt.title("Coverage Distribution (t=0.7)")

plt.legend()

plt.savefig("coverage_histogram.png")
plt.close()

# BOXPLOT

plt.figure(figsize=(8, 6))

box_data = [
    df[title_col],
    df[description_col]
]

labels = [
    "Title",
    "Description"
]

bp = plt.boxplot(
    box_data,
    labels=labels,
    patch_artist=True
)

# -------- Add median labels --------
medians = [
    df[title_col].median(),
    df[description_col].median()
]

for i, median in enumerate(medians, start=1):

    plt.text(
        i + 0.08,
        median,
        f"{median:.2f}",
        verticalalignment='center',
        fontsize=9
    )

plt.ylabel("Coverage")
plt.title("Coverage Comparison (t=0.7)")

plt.savefig(
    "coverage_boxplot.png",
    bbox_inches='tight'
)

plt.close()


# THRESHOLD CATEGORIES

def categorize(score):
    if score >= 0.75:
        return "High"
    elif score >= 0.5:
        return "Moderate"
    else:
        return "Low"

# -------- Title --------
df["title_coverage_category"] = df[title_col].apply(categorize)

# -------- Description --------
df["description_coverage_category"] = df[description_col].apply(categorize)

title_category_counts = (
    df["title_coverage_category"]
    .value_counts(normalize=True) * 100
)

description_category_counts = (
    df["description_coverage_category"]
    .value_counts(normalize=True) * 100
)

print("\n📊 TITLE COVERAGE CATEGORY DISTRIBUTION (%)\n")
print(title_category_counts)

print("\n📊 DESCRIPTION COVERAGE CATEGORY DISTRIBUTION (%)\n")
print(description_category_counts)


# RANGE DISTRIBUTION

print("\n📊 COVERAGE RANGE DISTRIBUTION (0–1)\n")

bins = [i/10 for i in range(11)]

labels = [
    f"{bins[i]:.1f}-{bins[i+1]:.1f}"
    for i in range(len(bins)-1)
]

# -------- Title bins --------
df["title_coverage_bin"] = pd.cut(
    df[title_col],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# -------- Description bins --------
df["description_coverage_bin"] = pd.cut(
    df[description_col],
    bins=bins,
    labels=labels,
    include_lowest=True
)

title_bin_counts = (
    df["title_coverage_bin"]
    .value_counts()
    .sort_index()
)

description_bin_counts = (
    df["description_coverage_bin"]
    .value_counts()
    .sort_index()
)

title_bin_percent = (
    title_bin_counts / len(df)
) * 100

description_bin_percent = (
    description_bin_counts / len(df)
) * 100

range_df = pd.DataFrame({
    "Range": title_bin_counts.index,

    "Title Percentage": title_bin_percent.values,

    "Description Percentage":
        description_bin_percent.values
})

print(range_df)


# RANGE BAR PLOT

x = np.arange(len(labels))
width = 0.35

plt.figure()

plt.bar(
    x - width/2,
    title_bin_percent.values,
    width,
    label="Title"
)

plt.bar(
    x + width/2,
    description_bin_percent.values,
    width,
    label="Description"
)

plt.xticks(x, labels, rotation=45)

plt.xlabel("Coverage Range")
plt.ylabel("Percentage of Datasets")

plt.title("Coverage Distribution (0–1 Range)")

plt.legend()

plt.tight_layout()

plt.savefig("coverage_range_distribution.png")
plt.close()


# MULTI-THRESHOLD COMPARISON

print("\n📊 MULTI-THRESHOLD COVERAGE COMPARISON\n")

title_threshold_means = df[[
    "title_coverage_0.6",
    "title_coverage_0.7",
    "title_coverage_0.8"
]].mean()

description_threshold_means = df[[
    "description_coverage_0.6",
    "description_coverage_0.7",
    "description_coverage_0.8"
]].mean()

print("\nTitle Coverage Means\n")
print(title_threshold_means)

print("\nDescription Coverage Means\n")
print(description_threshold_means)

# -------- ROBUSTNESS TABLE --------
robustness_df = pd.DataFrame({
    "Threshold": [0.6, 0.7, 0.8],
    "Title Coverage Mean": title_threshold_means.values,
    "Description Coverage Mean": description_threshold_means.values
})

print("\n📊 ROBUSTNESS SUMMARY TABLE\n")
print(robustness_df)

# Plot threshold comparison
plt.figure()

plt.plot(
    ["0.6", "0.7", "0.8"],
    title_threshold_means.values,
    marker='o',
    label="Title"
)

plt.plot(
    ["0.6", "0.7", "0.8"],
    description_threshold_means.values,
    marker='o',
    label="Description"
)

plt.xlabel("Similarity Threshold")
plt.ylabel("Average Coverage")

plt.title("Coverage vs Similarity Threshold")

plt.legend()

plt.savefig("coverage_threshold_comparison.png")
plt.close()

# SCATTER: TITLE VS DESCRIPTION

plt.figure()

plt.scatter(
    df[title_col],
    df[description_col],
    alpha=0.5
)

plt.xlabel("Title Coverage")
plt.ylabel("Description Coverage")

plt.title("Title vs Description Coverage")

plt.savefig("scatter_title_vs_description_coverage.png")
plt.close()


# SCATTER: KEYWORD COUNT VS COVERAGE

plt.figure()

plt.scatter(
    df["num_keywords"],
    df[title_col],
    alpha=0.5,
    label="Title"
)

plt.scatter(
    df["num_keywords"],
    df[description_col],
    alpha=0.5,
    label="Description"
)

plt.xlabel("Number of Keywords")
plt.ylabel("Coverage")

plt.title("Keyword Count vs Coverage")

plt.legend()

plt.savefig("coverage_vs_keywords.png")
plt.close()


# CORRELATION

print("\n🔗 CORRELATION MATRIX\n")

correlation = df[[
    "title_coverage_0.7",
    "description_coverage_0.7",
    "avg_title_max_similarity",
    "avg_description_max_similarity",
    "num_keywords",
    "num_title_concepts",
    "num_description_concepts"
]].corr()

print(correlation)


# TOP & BOTTOM DATASETS

top_title = df.sort_values(
    title_col,
    ascending=False
).head(10)

bottom_title = df.sort_values(
    title_col,
    ascending=True
).head(10)

top_description = df.sort_values(
    description_col,
    ascending=False
).head(10)

bottom_description = df.sort_values(
    description_col,
    ascending=True
).head(10)

print("\n🏆 TOP 10 TITLE COVERAGE DATASETS\n")
print(top_title)

print("\n⚠️ BOTTOM 10 TITLE COVERAGE DATASETS\n")
print(bottom_title)

print("\n🏆 TOP 10 DESCRIPTION COVERAGE DATASETS\n")
print(top_description)

print("\n⚠️ BOTTOM 10 DESCRIPTION COVERAGE DATASETS\n")
print(bottom_description)


# SAVE TABLES

summary_df.to_csv(
    "coverage_summary_statistics.csv",
    index=False
)

range_df.to_csv(
    "coverage_range_distribution.csv",
    index=False
)

robustness_df.to_csv(
    "coverage_robustness_summary.csv",
    index=False
)

title_category_counts.to_csv(
    "title_coverage_category_distribution.csv"
)

description_category_counts.to_csv(
    "description_coverage_category_distribution.csv"
)

print("\n💾 Files saved:")
print("- coverage_summary_statistics.csv")
print("- coverage_range_distribution.csv")
print("- coverage_robustness_summary.csv")
print("- title_coverage_category_distribution.csv")
print("- description_coverage_category_distribution.csv")
print("- coverage_histogram.png")
print("- coverage_boxplot.png")
print("- coverage_range_distribution.png")
print("- coverage_threshold_comparison.png")
print("- scatter_title_vs_description_coverage.png")
print("- coverage_vs_keywords.png")

print("\n✅ Coverage analysis complete!")


✅ Loaded coverage results
                                          dataset_id  num_keywords  \
0  http://data.europa.eu/88u/dataset/taxi-and-pri...            12   
1  http://data.europa.eu/88u/dataset/qics-data-2d...             6   
2  http://data.europa.eu/88u/dataset/free-school-...             9   
3  http://data.europa.eu/88u/dataset/farm-census-...            14   
4  http://data.europa.eu/88u/dataset/register-of-...             9   

   num_title_concepts  num_description_concepts  avg_title_max_similarity  \
0                   5                         5                  0.774895   
1                   5                         5                  0.387100   
2                   5                         5                  0.840352   
3                   5                         5                  0.745399   
4                   5                         5                  0.807681   

   avg_description_max_similarity  title_coverage_0.6  title_coverage_0.7  \
0           

C:\Users\deper\AppData\Local\Temp\ipykernel_18648\1956532859.py:102: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = plt.boxplot(



📊 TITLE COVERAGE CATEGORY DISTRIBUTION (%)

title_coverage_category
Low         75.61
Moderate    12.77
High        11.62
Name: proportion, dtype: float64

📊 DESCRIPTION COVERAGE CATEGORY DISTRIBUTION (%)

description_coverage_category
Low         88.37
Moderate     7.39
High         4.24
Name: proportion, dtype: float64

📊 COVERAGE RANGE DISTRIBUTION (0–1)

     Range  Title Percentage  Description Percentage
0  0.0-0.1             36.24                   42.72
1  0.1-0.2             21.31                   25.43
2  0.2-0.3              1.05                    1.77
3  0.3-0.4             17.01                   18.45
4  0.4-0.5              1.09                    0.09
5  0.5-0.6              9.55                    7.25
6  0.6-0.7              2.13                    0.05
7  0.7-0.8              7.03                    3.43
8  0.8-0.9              0.00                    0.00
9  0.9-1.0              4.59                    0.81

📊 MULTI-THRESHOLD COVERAGE COMPARISON


Title Coverage